In [ ]:
!pip install numpy scipy

## **Kiểm chứng kết quả bằng Scipy / Numpy.**
Ở trong đây, với mỗi phần nhóm sẽ trình bày khoảng 5-8 test cases, có khả năng bao phủ hết các trường hợp có thể xảy ra của mỗi hàm (happy case, edge case, ...)

## **1. Kiểm chứng Gaussian elimination + Back substitution**

Mỗi test dưới đây đều có **happy case** và **edge case** (pivot 0, suy biến, vô nghiệm, vô số nghiệm, ma trận chữ nhật...).


## **1.1 Kiểm chứng bằng toán học và các hàm có sẵn**

In [ ]:
from gaussian import gaussian_eliminate, back_substitution

def _close_to(a: float, b: float, eps: float = 1e-9) -> bool:
    """Kiểm tra xem hai số thực có gần với nhau tại một ngưỡng epsilon cho trước."""
    return abs(a - b) <= eps


def _vector_close_to(v1, v2, eps: float = 1e-9) -> bool:
    """Kiểm tra xem hai vector có gần với nhau tại một ngưỡng epsilon cho trước."""
    return len(v1) == len(v2) and all(_close_to(a, b, eps) for a, b in zip(v1, v2))


def _mul_Ax(A, x):
    return [sum(A[i][j] * x[j] for j in range(len(x))) for i in range(len(A))]


def output(name, ok, desc=""):
    trang_thai = "ĐẠT" if ok else "KHÔNG ĐẠT"
    print(f"{name}: {trang_thai} {desc}")

In [ ]:
# Test 1: Happy case, hệ 2x2 có nghiệm duy nhất
A = [[2.0, 1.0], [1.0, 3.0]]
b = [1.0, 2.0]
U, c, _ = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (status == "Hệ có nghiệm duy nhất") and _vector_close_to(_mul_Ax([[2.0, 1.0], [1.0, 3.0]], x), b)
output("Test 1", ok, f"status={status}, x={x}")


# Test 2: Happy case, cần partial pivoting (pivot ban đầu = 0)
A = [[0.0, 1.0], [2.0, 3.0]]
b = [1.0, 5.0]
U, c, n_swaps = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (n_swaps >= 1) and (status == "Hệ có nghiệm duy nhất") and _vector_close_to(_mul_Ax([[0.0, 1.0], [2.0, 3.0]], x), b)
output("Test 2", ok, f"swap={n_swaps}, status={status}, x={x}")


# Test 3: Edge case, ma trận suy biến nhưng vẫn có vô số nghiệm
A = [[1.0, 1.0], [2.0, 2.0]]
b = [2.0, 4.0]
U, c, _ = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (status == "Hệ có vô số nghiệm")
output("Test 3", ok, f"status={status}")


# Test 4: Edge case, vô nghiệm
A = [[1.0, 1.0], [1.0, 1.0]]
b = [1.0, 2.0]
U, c, _ = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (status == "Hệ vô nghiệm")
output("Test 4", ok, f"status={status}")


# Test 5: Edge case, ma trận chữ nhật m > n (hệ thừa phương) nhưng tương thích
A = [[1.0, 1.0], [1.0, -1.0], [2.0, 0.0]]
b = [2.0, 0.0, 2.0]
U, c, _ = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (status == "Hệ có nghiệm duy nhất") and _vector_close_to(_mul_Ax([[1.0, 1.0], [1.0, -1.0], [2.0, 0.0]], x), b)
output("Test 5", ok, f"status={status}, x={x}")


# Test 6: Edge case, ma trận chữ nhật m > n nhưng không tương thích
A = [[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]]
b = [0.0, 0.0, 1.0]
U, c, _ = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (status == "Hệ vô nghiệm")
output("Test 6", ok, f"status={status}")


# Test 7: Edge case, ma trận chữ nhật m < n (thiếu phương) → vô số nghiệm nếu tương thích
A = [[1.0, 2.0, 3.0], [0.0, 1.0, 4.0]]
b = [1.0, 2.0]
U, c, _ = gaussian_eliminate(A, b)
x, status = back_substitution(U, c)
ok = (status == "Hệ có vô số nghiệm")
output("Test 7", ok, f"status={status}")


# Test 8: Edge case, kiểm tra only_one=True (không ghép b)
A = [[1.0, 2.0], [3.0, 4.0]]
U, c, _ = gaussian_eliminate(A, only_one=True)
ok = (c == []) and (len(U) == 2) and (len(U[0]) == 2)
output("Test 8", ok, f"c={c}, U={U}")


Test 1: ĐẠT status=Hệ có nghiệm duy nhất, x=[0.2, 0.6]
Test 2: ĐẠT swap=1, status=Hệ có nghiệm duy nhất, x=[1.0, 1.0]
Test 3: ĐẠT status=Hệ có vô số nghiệm
Test 4: ĐẠT status=Hệ vô nghiệm
Test 5: ĐẠT status=Hệ có nghiệm duy nhất, x=[1.0, 1.0]
Test 6: ĐẠT status=Hệ vô nghiệm
Test 7: ĐẠT status=Hệ có vô số nghiệm
Test 8: ĐẠT c=[], U=[[3.0, 4.0], [0.0, 0.6666666666666667]]


## **1.2. Đối chiếu với NumPy/SciPy**

Mục này dùng NumPy (và SciPy nếu có) để đối chiếu các **case có nghiệm duy nhất**.
Với các case vô nghiệm / vô số nghiệm, ta đối chiếu bằng tiêu chí nhất quán (residual).


In [6]:
import numpy as np

def numpy_solve_unique(A, b):
    """Giải hệ vuông nghiệm duy nhất bằng NumPy; ném lỗi nếu không giải được."""
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float)
    x_np = np.linalg.solve(A_np, b_np)
    return x_np.tolist()


def residual_norm(A, x, b):
    """Tính chuẩn 2 của residual ||Ax-b||_2."""
    A_np = np.array(A, dtype=float)
    x_np = np.array(x, dtype=float)
    b_np = np.array(b, dtype=float)
    r = A_np @ x_np - b_np
    return float(np.linalg.norm(r, ord=2))


# Nếu có SciPy thì dùng thêm để kiểm tra (không bắt buộc)
# Lưu ý: dùng importlib để tránh IDE báo lỗi khi chưa cài SciPy.
import importlib

try:
    la = importlib.import_module("scipy.linalg")

    def scipy_solve_unique(A, b):
        """Giải hệ vuông nghiệm duy nhất bằng SciPy."""
        A_np = np.array(A, dtype=float)
        b_np = np.array(b, dtype=float)
        x_sp = la.solve(A_np, b_np, assume_a="gen")
        return x_sp.tolist()

    has_scipy = True
except Exception:
    has_scipy = False


# Đối chiếu cho các case nghiệm duy nhất (Test 1, 2, 5)
# Test 1
A1 = [[2.0, 1.0], [1.0, 3.0]]
b1 = [1.0, 2.0]
U, c, _ = gaussian_eliminate(A1, b1)
x, status = back_substitution(U, c)

x_np = numpy_solve_unique(A1, b1)
ok = (status == "Hệ có nghiệm duy nhất") and _vector_close_to(x, x_np)
output("Đối chiếu NumPy - Test 1", ok, f"x={x}, x_np={x_np}, residual={residual_norm(A1, x, b1):.2e}")

if has_scipy:
    x_sp = scipy_solve_unique(A1, b1)
    ok_sp = _vector_close_to(x, x_sp)
    output("Đối chiếu SciPy - Test 1", ok_sp, f"x_sp={x_sp}")


# Test 2
A2 = [[0.0, 1.0], [2.0, 3.0]]
b2 = [1.0, 5.0]
U, c, _ = gaussian_eliminate(A2, b2)
x, status = back_substitution(U, c)

x_np = numpy_solve_unique(A2, b2)
ok = (status == "Hệ có nghiệm duy nhất") and _vector_close_to(x, x_np)
output("Đối chiếu NumPy - Test 2", ok, f"x={x}, x_np={x_np}, residual={residual_norm(A2, x, b2):.2e}")

if has_scipy:
    x_sp = scipy_solve_unique(A2, b2)
    ok_sp = _vector_close_to(x, x_sp)
    output("Đối chiếu SciPy - Test 2", ok_sp, f"x_sp={x_sp}")


# Test 5 (hệ thừa phương nhưng tương thích): đối chiếu bằng residual và nghiệm theo least-squares
A5 = [[1.0, 1.0], [1.0, -1.0], [2.0, 0.0]]
b5 = [2.0, 0.0, 2.0]
U, c, _ = gaussian_eliminate(A5, b5)
x, status = back_substitution(U, c)

res = residual_norm(A5, x, b5) if status == "Hệ có nghiệm duy nhất" else float("inf")
# Numpy least squares để kiểm tra residual tối thiểu
x_ls, *_ = np.linalg.lstsq(np.array(A5, float), np.array(b5, float), rcond=None)
res_ls = float(np.linalg.norm(np.array(A5, float) @ x_ls - np.array(b5, float)))

ok = (status == "Hệ có nghiệm duy nhất") and (res <= 1e-9) and (res_ls <= 1e-9)
output(
    "Đối chiếu NumPy - Test 5",
    ok,
    f"status={status}, residual={res:.2e}, residual_lstsq={res_ls:.2e}, x={x}, x_lstsq={x_ls.tolist()}"
)


Đối chiếu NumPy - Test 1: ĐẠT x=[0.2, 0.6], x_np=[0.2, 0.6], residual=2.22e-16
Đối chiếu SciPy - Test 1: ĐẠT x_sp=[0.2, 0.6]
Đối chiếu NumPy - Test 2: ĐẠT x=[1.0, 1.0], x_np=[1.0, 1.0], residual=0.00e+00
Đối chiếu SciPy - Test 2: ĐẠT x_sp=[1.0, 1.0]
Đối chiếu NumPy - Test 5: ĐẠT status=Hệ có nghiệm duy nhất, residual=0.00e+00, residual_lstsq=1.28e-15, x=[1.0, 1.0], x_lstsq=[1.0000000000000004, 1.0000000000000002]


## **2. Kiểm chứng `determinant`**

Mỗi test dưới đây có cả **happy case** và **edge case**.


## **2.1 Kiểm chứng bằng toán học, và các hàm có sẵn**

In [9]:
from determinant import determinant

# Test 1: Happy case, ma trận đơn vị
A = [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
val = determinant(A)
output("det Test 1", _close_to(val, 1.0), f"det={val}")


# Test 2: Happy case, ma trận tam giác trên (det = tích đường chéo)
A = [[2.0, 3.0, 4.0], [0.0, -1.0, 5.0], [0.0, 0.0, 0.5]]
val = determinant(A)
output("det Test 2", _close_to(val, 2.0 * (-1.0) * 0.5), f"det={val}")


# Test 3: Edge case, hoán vị 2 hàng làm đổi dấu
A = [[1.0, 2.0], [3.0, 4.0]]
A_swapped = [[3.0, 4.0], [1.0, 2.0]]
val = determinant(A)
val2 = determinant(A_swapped)
ok = _close_to(val2, -val)
output("det Test 3", ok, f"det(A)={val}, det(swap)={val2}")


# Test 4: Edge case, ma trận suy biến (det = 0)
A = [[1.0, 2.0], [2.0, 4.0]]
val = determinant(A)
output("det Test 4", _close_to(val, 0.0), f"det={val}")


# Test 5: Edge case, ma trận 1x1
A = [[-7.0]]
val = determinant(A)
output("det Test 5", _close_to(val, -7.0), f"det={val}")


# Test 6: Edge case, ma trận không vuông -> phải báo lỗi
ok = False
try:
    determinant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
except ValueError:
    ok = True
output("det Test 6", ok, "ma trận không vuông")


# Test 7: Edge case, ma trận hàng không đều -> phải báo lỗi
ok = False
try:
    determinant([[1.0, 2.0], [3.0]])
except ValueError:
    ok = True
output("det Test 7", ok, "ma trận hàng không đều")


det Test 1: ĐẠT det=1.0
det Test 2: ĐẠT det=-1.0
det Test 3: ĐẠT det(A)=-2.0, det(swap)=2.0
det Test 4: ĐẠT det=-0.0
det Test 5: ĐẠT det=-7.0
det Test 6: ĐẠT ma trận không vuông
det Test 7: ĐẠT ma trận hàng không đều


## **2.2 Đối chiếu `determinant` với NumPy/SciPy**

So sánh `determinant(A)` với `numpy.linalg.det(A)` và `scipy.linalg.det(A)` (nếu có SciPy).


In [10]:
import numpy as np

# SciPy optional (dùng importlib để tránh lỗi IDE)
try:
    la_det = importlib.import_module("scipy.linalg")
    has_scipy_det = True
except Exception:
    has_scipy_det = False


def compare_det(A, name: str):
    """Đối chiếu định thức giữa code của mình và NumPy/SciPy."""
    det_ours = float(determinant(A))
    det_np = float(np.linalg.det(np.array(A, dtype=float)))
    ok = _close_to(det_ours, det_np, eps=1e-7 * max(1.0, abs(det_np)))
    output(name + " (NumPy)", ok, f"ours={det_ours:.6g}, numpy={det_np:.6g}")

    if has_scipy_det:
        det_sp = float(la_det.det(np.array(A, dtype=float)))
        ok_sp = _close_to(det_ours, det_sp, eps=1e-7 * max(1.0, abs(det_sp)))
        output(name + " (SciPy)", ok_sp, f"scipy={det_sp:.6g}")


# Đối chiếu một số ma trận tiêu biểu
compare_det([[1.0, 0.0], [0.0, 1.0]], "Đối chiếu det - I2")
compare_det([[1.0, 2.0], [3.0, 4.0]], "Đối chiếu det - 2x2")
compare_det([[2.0, 3.0, 4.0], [0.0, -1.0, 5.0], [0.0, 0.0, 0.5]], "Đối chiếu det - tam giác")

# Random: kiểm tra tương đối trên vài ma trận ngẫu nhiên
rng = np.random.default_rng(0)
ok_all = True
for k in range(5):
    A = rng.normal(size=(4, 4))
    det_ours = float(determinant(A.tolist()))
    det_np = float(np.linalg.det(A))
    if not _close_to(det_ours, det_np, eps=1e-7 * max(1.0, abs(det_np))):
        ok_all = False
        break
output("Đối chiếu det - random 5 mẫu", ok_all)


Đối chiếu det - I2 (NumPy): ĐẠT ours=1, numpy=1
Đối chiếu det - I2 (SciPy): ĐẠT scipy=1
Đối chiếu det - 2x2 (NumPy): ĐẠT ours=-2, numpy=-2
Đối chiếu det - 2x2 (SciPy): ĐẠT scipy=-2
Đối chiếu det - tam giác (NumPy): ĐẠT ours=-1, numpy=-1
Đối chiếu det - tam giác (SciPy): ĐẠT scipy=-1
Đối chiếu det - random 5 mẫu: ĐẠT 


## 3. **Kiểm chứng `inverse`**

Mỗi test dưới đây có cả **happy case** và **edge case**.


## **3.1 Kiểm chứng bằng toán học và các hàm có sẵn**

In [14]:
from inverse import inverse

def matmul(A, B):
    """Nhân hai ma trận A (m×n) và B (n×p)."""
    m = len(A)
    n = len(A[0])
    p = len(B[0])
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(p)] for i in range(m)]


def eye(n: int):
    """Tạo ma trận đơn vị n×n."""
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def fro_norm(M):
    """Chuẩn Frobenius của ma trận."""
    return sum(x * x for row in M for x in row) ** 0.5

#  Kiểm tra tính đúng đắn của hàm `inverse` qua nhiều trường hợp khác nhau:
# - "Happy case": các ma trận thường gặp, lẻ hoặc có phần tử đặc biệt (như phần tử chéo chính bằng 0), 
#   kiểm tra xem nghịch đảo thu được nhân lại với ma trận gốc có ra gần đúng ma trận đơn vị không 
#   (dùng chuẩn Frobenius để đánh giá sai số).
# - "Edge case": các trường hợp biên như ma trận 1x1, ma trận suy biến (không khả nghịch), ma trận không vuông, 
#   hoặc các ma trận có hàng/ cột không đều, kiểm tra hàm có xử lý đúng ngoại lệ và báo lỗi thích hợp không.
# Các output sẽ thông báo kết quả kiểm thử và giá trị sai số (nếu có) để tiện theo dõi.

# Test 1: Happy case, nghịch đảo của I
A = eye(3)
Ai = inverse(A)
err = fro_norm([[Ai[i][j] - A[i][j] for j in range(3)] for i in range(3)])
output("inv Test 1", err < 1e-12, f"||A^{-1}-I||_F={err:.2e}")


# Test 2: Happy case, ma trận 2x2 cơ bản
A = [[1.0, 2.0], [3.0, 4.0]]
Ai = inverse(A)
I_hat = matmul(Ai, A)
err = fro_norm([[I_hat[i][j] - (1.0 if i == j else 0.0) for j in range(2)] for i in range(2)])
output("inv Test 2", err < 1e-9, f"||A^{-1}A-I||_F={err:.2e}")


# Test 3: Happy case, pivot ban đầu = 0 nhưng vẫn khả nghịch
A = [[0.0, 1.0], [2.0, 3.0]]
Ai = inverse(A)
I_hat = matmul(Ai, A)
err = fro_norm([[I_hat[i][j] - (1.0 if i == j else 0.0) for j in range(2)] for i in range(2)])
output("inv Test 3", err < 1e-9, f"||A^{-1}A-I||_F={err:.2e}")


# Test 4: Edge case, ma trận 1x1
A = [[-7.0]]
Ai = inverse(A)
ok = _close_to(Ai[0][0], -1.0 / 7.0)
output("inv Test 4", ok, f"A^{-1}={Ai}")


# Test 5: Edge case, suy biến -> phải báo lỗi
ok = False
try:
    inverse([[1.0, 2.0], [2.0, 4.0]])
except ValueError:
    ok = True
output("inv Test 5", ok, "ma trận suy biến")


# Test 6: Edge case, không vuông -> phải báo lỗi
ok = False
try:
    inverse([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
except ValueError:
    ok = True
output("inv Test 6", ok, "ma trận không vuông")


# Test 7: Edge case, hàng không đều -> phải báo lỗi
ok = False
try:
    inverse([[1.0, 2.0], [3.0]])
except ValueError:
    ok = True
output("inv Test 7", ok, "ma trận hàng không đều")


inv Test 1: ĐẠT ||A^-1-I||_F=0.00e+00
inv Test 2: ĐẠT ||A^-1A-I||_F=5.44e-16
inv Test 3: ĐẠT ||A^-1A-I||_F=0.00e+00
inv Test 4: ĐẠT A^-1=[[-0.14285714285714285]]
inv Test 5: ĐẠT ma trận suy biến
inv Test 6: ĐẠT ma trận không vuông
inv Test 7: ĐẠT ma trận hàng không đều


## 3.2 **Đối chiếu `inverse` với NumPy/SciPy**

So sánh `inverse(A)` với `numpy.linalg.inv(A)` và `scipy.linalg.inv(A)` (nếu có SciPy).
Tiêu chí kiểm tra: chuẩn của \(A^{-1}A - I\) và sai khác giữa hai ma trận nghịch đảo.


In [15]:
import numpy as np

try:
    la_inv = importlib.import_module("scipy.linalg")
    has_scipy_inv = True
except Exception:
    has_scipy_inv = False


def compare_inverse(A, name: str):
    """Đối chiếu nghịch đảo giữa code của mình và NumPy/SciPy."""
    A_np = np.array(A, dtype=float)

    Ai_ours = np.array(inverse(A), dtype=float)
    Ai_np = np.linalg.inv(A_np)

    err_ours = float(np.linalg.norm(Ai_ours @ A_np - np.eye(A_np.shape[0])))
    err_diff = float(np.linalg.norm(Ai_ours - Ai_np))

    ok = (err_ours < 1e-8) and (err_diff < 1e-6)
    output(name + " (NumPy)", ok, f"||A^{-1}A-I||={err_ours:.2e}, ||ours-np||={err_diff:.2e}")

    if has_scipy_inv:
        Ai_sp = la_inv.inv(A_np)
        err_diff_sp = float(np.linalg.norm(Ai_ours - Ai_sp))
        ok_sp = err_diff_sp < 1e-6
        output(name + " (SciPy)", ok_sp, f"||ours-sp||={err_diff_sp:.2e}")


compare_inverse([[1.0, 2.0], [3.0, 4.0]], "Đối chiếu inv - 2x2")
compare_inverse([[0.0, 1.0], [2.0, 3.0]], "Đối chiếu inv - pivot")

# Random invertible: kiểm tra trên vài ma trận ngẫu nhiên (tránh suy biến)
rng = np.random.default_rng(1)
ok_all = True
for k in range(5):
    A = rng.normal(size=(4, 4))
    if abs(np.linalg.det(A)) < 1e-3:
        continue
    try:
        Ai_ours = np.array(inverse(A.tolist()), dtype=float)
    except ValueError:
        continue
    err = float(np.linalg.norm(Ai_ours @ A - np.eye(4)))
    if err > 1e-7:
        ok_all = False
        break
output("Đối chiếu inv - random 5 mẫu", ok_all)


Đối chiếu inv - 2x2 (NumPy): ĐẠT ||A^-1A-I||=5.47e-16, ||ours-np||=3.19e-16
Đối chiếu inv - 2x2 (SciPy): ĐẠT ||ours-sp||=3.19e-16
Đối chiếu inv - pivot (NumPy): ĐẠT ||A^-1A-I||=0.00e+00, ||ours-np||=0.00e+00
Đối chiếu inv - pivot (SciPy): ĐẠT ||ours-sp||=0.00e+00
Đối chiếu inv - random 5 mẫu: ĐẠT 
